# LeetCode 2027 — Minimum Moves to Convert String bằng Python

**Link đề:** https://leetcode.com/problems/minimum-moves-to-convert-string/
**Độ khó:** Easy · **Chủ đề:** String, Greedy

```python
class Solution:
    def minimumMoves(self, s: str) -> int:

```

## Mục tiêu học
1. Hiểu đề: mỗi **move** biến 3 ký tự **liên tiếp** thành `'O'`; tìm số move ít nhất để mọi ký tự thành `'O'`.
2. Nắm `while` với chỉ số tự điều khiển (`i += 3` / `i += 1`), `list(s)` để mô phỏng, `str.count`, `re.findall`.
3. Hiểu **tham lam (greedy)**: gặp `'X'` trái nhất chưa xử lý thì đặt move **bắt đầu ngay tại đó** ("trì hoãn move càng lâu càng tốt").
4. Chứng minh greedy tối ưu bằng **lập luận trao đổi (exchange argument)**; kiểm chứng bằng **quy hoạch động** và **BFS vét cạn**.
5. Trace, so sánh 5 cách, test đối chiếu, bài tập biến thể (cửa sổ độ dài `k`).

## 1. Đề bài yêu cầu gì?

Chuỗi `s` gồm `n` ký tự, mỗi ký tự là `'X'` hoặc `'O'`.

Một **move**: chọn **3 ký tự liên tiếp** và biến chúng thành `'O'` (ký tự đã là `'O'` thì vẫn là `'O'`).

Trả về số move **ít nhất** để mọi ký tự thành `'O'`.

| `s` | Kết quả | Một cách làm tối ưu |
|---|:-:|---|
| `"XXX"` | `1` | `[XXX]` → `OOO` |
| `"XXOX"` | `2` | `[XXO]X` → `OOOX`, rồi `O[OOX]` → `OOOO` |
| `"OOOO"` | `0` | không có X |

**Constraints:** `3 <= s.length <= 1000`, `s[i]` là `'X'` hoặc `'O'`.

Chú ý: một move luôn phủ **đúng 3** vị trí liên tiếp **nằm trong** chuỗi (không vượt quá cuối). Vì `n ≥ 3` nên luôn có cách phủ mọi vị trí.

In [1]:
s = "XXOX"
print("chỉ số:", " ".join(str(i) for i in range(len(s))))
print("ký tự :", " ".join(s))
print("vị trí các X:", [i for i, c in enumerate(s) if c == 'X'])

chỉ số: 0 1 2 3
ký tự : X X O X
vị trí các X: [0, 1, 3]


## 2. Nền tảng Python cần biết

### 2.1. Chuỗi bất biến → mô phỏng bằng `list`

Để "biến 3 ký tự thành O" ta cần sửa chuỗi. `str` không sửa được, nên đổi sang `list(s)`, sửa, rồi `"".join(...)` để xem.

In [2]:
chars = list("XXOX")
chars[0:3] = ['O', 'O', 'O']        # gán vào slice: thay 3 phần tử
print("".join(chars))

OOOX


### 2.2. `while` với bước nhảy thay đổi

`for i in range(n)` luôn tăng 1. Bài này cần **nhảy 3** khi thực hiện move, **nhảy 1** khi gặp `'O'` → dùng `while i < n` và tự cập nhật `i`.

In [3]:
i, n = 0, 10
path = []
while i < n:
    path.append(i)
    i += 3 if i % 2 == 0 else 1      # ví dụ bước nhảy tùy điều kiện
print("các i đã đi qua:", path)

các i đã đi qua: [0, 3, 4, 7, 8]


### 2.3. `min(i + 3, n)` — chặn biên khi gán slice

Khi mô phỏng move bắt đầu ở `i` gần cuối, `i + 3` có thể vượt `n`. Với slicing Python **không lỗi** (tự cắt ngắn), nhưng hiểu rõ giúp tránh nhầm khi đếm.

In [4]:
s = "OOX"
print(s[2:5], "| độ dài:", len(s[2:5]))   # chỉ còn 1 ký tự

X | độ dài: 1


### 2.4. Biểu thức chính quy (regex) `re.findall`

Module `re` tìm mẫu trong chuỗi. `X.{0,2}` nghĩa là: một ký tự `X`, theo sau là **0 đến 2** ký tự bất kỳ (`.`). `re.findall` tìm các khớp **không chồng nhau, từ trái sang phải** — ta sẽ thấy nó trùng với greedy!

In [5]:
import re
print(re.findall(r"X.{0,2}", "XXOX"))
print(re.findall(r"X.{0,2}", "OXOOXXOOX"))

['XXO', 'X']
['XOO', 'XXO', 'X']


## 3. Hiểu bài bằng tay: thử vài cách đặt move

`s = "XXOX"` (n = 4). Các vị trí bắt đầu move hợp lệ: `0` (phủ 0..2) hoặc `1` (phủ 1..3).

| Cách | Move | Sau move | Còn X? |
|---|---|---|---|
| A | bắt đầu 0 | `OOOX` | còn X ở 3 → cần thêm move bắt đầu 1 → **2 move** |
| B | bắt đầu 1 | `XOOO` | còn X ở 0 → cần thêm move bắt đầu 0 → **2 move** |

Không có move nào phủ được cả vị trí 0 và 3 (cách nhau 4 > 3) → tối thiểu **2**.

Với `s = "OXOOXO"`? X ở 1 và 4, cách nhau 3 vị trí (1, 2, 3, 4 là 4 ô) → không phủ chung → **2**.

## 4. Cách 1 — Baseline: mô phỏng thật trên list

Ý tưởng trực giác: lặp lại "tìm `'X'` đầu tiên, đặt move bắt đầu tại đó (hoặc lùi lại nếu sát cuối), biến 3 ô thành `'O'`" cho đến khi hết `'X'`.

Dùng `chars.index('X')` để tìm X đầu tiên mỗi lần → mỗi lần `O(n)`.

In [6]:
def min_moves_simulate(s):
    chars = list(s)
    n = len(chars)
    moves = 0
    while 'X' in chars:
        i = chars.index('X')         # X trái nhất
        start = min(i, n - 3)        # move phải nằm trong chuỗi
        for k in range(start, start + 3):
            chars[k] = 'O'
        moves += 1
    return moves

print(min_moves_simulate("XXX"))    # 1
print(min_moves_simulate("XXOX"))   # 2
print(min_moves_simulate("OOOO"))   # 0

1
2
0


### Nút thắt của mô phỏng

- Mỗi vòng `while` chạy `'X' in chars` và `chars.index('X')` → `O(n)`; có thể tới `n/3` vòng → tổng `O(n²)`.
- Thực ra ta **không cần sửa chuỗi**: sau khi đặt move tại `i`, ta biết chắc `i, i+1, i+2` đã là `'O'` → chỉ cần **nhảy** con trỏ qua 3 ô đó.

Ngoài ra: `start = min(i, n − 3)` — khi X sát cuối ta lùi move lại. Nhưng nếu X trái nhất ở `i`, mọi ô trước `i` đã là O nên lùi lại cũng không lợi gì; quan trọng là move **phủ được** `i`. Ở cách tối ưu ta chỉ cần đếm, không cần biết move nằm chính xác đâu.

## 5. Insight — Tham lam: "trì hoãn move càng lâu càng tốt"

Hint 2 của đề: *"Try delaying a move as long as possible."*

Xét `'X'` **trái nhất** chưa được phủ, ở vị trí `i`. Mọi ô bên trái `i` đều đã là `'O'` (hoặc đã được phủ). **Bắt buộc** phải có một move phủ `i`. Trong các move phủ `i` (bắt đầu tại `i−2`, `i−1` hoặc `i`), move **bắt đầu tại `i`** phủ `{i, i+1, i+2}` — **xa về bên phải nhất**, không lãng phí ô nào cho phần bên trái (đã xong).

→ Quy tắc: gặp `'X'` ở `i` → `moves += 1`, `i += 3`. Gặp `'O'` → `i += 1`.

(Nếu `i + 2 ≥ n`, ta "tưởng tượng" move lùi lại để nằm trong chuỗi — vẫn phủ được `i` và các ô sau nó, nên **số move không đổi**.)

In [7]:
# So sánh ba lựa chọn phủ X trái nhất ở vị trí 2 trong "OOXOXO"
s = "OOXOXO"
for start in [0, 1, 2]:
    covered = set(range(start, start + 3))
    remaining = [i for i, c in enumerate(s) if c == 'X' and i not in covered]
    print(f"move bắt đầu {start} phủ {sorted(covered)} -> X còn lại: {remaining}")

move bắt đầu 0 phủ [0, 1, 2] -> X còn lại: [4]
move bắt đầu 1 phủ [1, 2, 3] -> X còn lại: [4]
move bắt đầu 2 phủ [2, 3, 4] -> X còn lại: []


Chỉ move bắt đầu tại `2` (đúng vị trí X trái nhất) phủ luôn được X ở vị trí 4.

## 6. Cách 2 — Greedy nhảy chỉ số (lời giải chính)

Giảng từng đoạn:

```python
moves = 0
i = 0
while i < len(s):
    if s[i] == 'X':     # (1) X trái nhất chưa phủ
        moves += 1      # (2) đặt move bắt đầu tại i
        i += 3          # (3) i, i+1, i+2 đã xong -> nhảy qua
    else:
        i += 1          # (4) 'O' thì bỏ qua
return moves
```

Không cần sửa chuỗi, không cần `list` — chỉ đọc.

In [8]:
def min_moves(s):
    moves = 0
    i = 0
    while i < len(s):
        if s[i] == 'X':
            moves += 1
            i += 3
        else:
            i += 1
    return moves

print(min_moves("XXX"))    # 1
print(min_moves("XXOX"))   # 2
print(min_moves("OOOO"))   # 0

1
2
0


## 7. TRACE từng bước

Ví dụ `s = "OXOOXXOOX"` (n = 9).

| i | s[i] | Quyết định | moves | i mới | Ô được phủ |
|:-:|:-:|---|:-:|:-:|---|
| 0 | O | bỏ qua | 0 | 1 | |
| 1 | X | move tại 1 | 1 | 4 | 1, 2, 3 |
| 4 | X | move tại 4 | 2 | 7 | 4, 5, 6 |
| 7 | O | bỏ qua | 2 | 8 | |
| 8 | X | move tại 8 (thực tế lùi về 6..8) | 3 | 11 | 8 |
| 11 | — | `i ≥ n` dừng | **3** | | |

Lưu ý: X ở vị trí 5 được phủ "miễn phí" bởi move tại 4 — ta **nhảy qua** nó.

In [9]:
def trace_min_moves(s):
    n = len(s)
    moves, i = 0, 0
    covered = ['.'] * n
    print(f"s = {s!r}, n = {n}")
    while i < n:
        if s[i] == 'X':
            moves += 1
            lo = min(i, n - 3)                # vị trí thực tế của move (lùi nếu sát cuối)
            for k in range(lo, lo + 3):
                covered[k] = str(moves)
            print(f"  i={i}: 'X' -> move #{moves} phủ [{lo}..{lo + 2}] | phủ: {''.join(covered)} | i -> {i + 3}")
            i += 3
        else:
            print(f"  i={i}: 'O' -> bỏ qua | i -> {i + 1}")
            i += 1
    print(f"  i = {i} >= n -> dừng. Kết quả: {moves}")
    return moves

trace_min_moves("OXOOXXOOX")

s = 'OXOOXXOOX', n = 9
  i=0: 'O' -> bỏ qua | i -> 1
  i=1: 'X' -> move #1 phủ [1..3] | phủ: .111..... | i -> 4
  i=4: 'X' -> move #2 phủ [4..6] | phủ: .111222.. | i -> 7
  i=7: 'O' -> bỏ qua | i -> 8
  i=8: 'X' -> move #3 phủ [6..8] | phủ: .11122333 | i -> 11
  i = 11 >= n -> dừng. Kết quả: 3


3

Cột "phủ" in số thứ tự move đã phủ mỗi ô (`.` là chưa phủ). Ở move #3, vì `i = 8 = n − 1`, move thực tế đặt ở `[6..8]` — chồng lên ô 6 đã được move #2 phủ, nhưng không sao: ta chỉ cần phủ X ở 8.

Thêm một trace với các X liên tiếp dài:

In [10]:
trace_min_moves("XXXXXXX")

s = 'XXXXXXX', n = 7
  i=0: 'X' -> move #1 phủ [0..2] | phủ: 111.... | i -> 3
  i=3: 'X' -> move #2 phủ [3..5] | phủ: 111222. | i -> 6
  i=6: 'X' -> move #3 phủ [4..6] | phủ: 1112333 | i -> 9
  i = 9 >= n -> dừng. Kết quả: 3


3

## 8. Code nộp LeetCode

In [11]:
class Solution:
    def minimumMoves(self, s: str) -> int:
        moves = i = 0
        while i < len(s):
            if s[i] == 'X':
                moves += 1
                i += 3
            else:
                i += 1
        return moves

for s, expected in [("XXX", 1), ("XXOX", 2), ("OOOO", 0)]:
    got = Solution().minimumMoves(s)
    print(repr(s), "->", got)
    assert got == expected

'XXX' -> 1
'XXOX' -> 2
'OOOO' -> 0


`moves = i = 0` là **gán chuỗi**: cả hai biến cùng nhận `0` (an toàn với số nguyên vì số là bất biến).

## 9. Cách 3 — Quy hoạch động (DP) để kiểm chứng greedy

Khi chưa tin greedy, ta có thể tính đáp án tối ưu bằng DP:

`dp[i]` = số move ít nhất để phủ mọi `'X'` ở các vị trí `≥ i`.
- `dp[i] = 0` nếu `i ≥ n`.
- Nếu `s[i] == 'O'`: `dp[i] = dp[i+1]` (không cần phủ `i`).
- Nếu `s[i] == 'X'`: phải có move phủ `i`; vì bên trái đã xử lý, move tốt nhất phủ `i..i+2` → `dp[i] = 1 + dp[i+3]`.

Công thức nhánh `'X'` ở trên vẫn **giả định** move bắt đầu tại `i`. Để DP không dựa vào greedy, ta cho move phủ `i` bắt đầu ở **mọi** vị trí hợp lệ `start ∈ {i−2, i−1, i}` (với `0 ≤ start ≤ n−3`) và lấy `min`:

$$dp[i] = \min_{start} \big(1 + dp[start + 3]\big)$$

(Các ô trước `i` đều đã là `'O'` nên move lùi về trái không phủ thêm X nào ở bên trái.) Kết quả DP luôn trùng greedy — một bằng chứng thực nghiệm. Để kiểm chứng **hoàn toàn độc lập** ta dùng BFS ở mục 10.

In [12]:
from functools import lru_cache

def min_moves_dp(s):
    n = len(s)

    @lru_cache(maxsize=None)
    def dp(i):
        if i >= n:
            return 0
        if s[i] == 'O':
            return dp(i + 1)
        # move phải phủ i, bắt đầu tại start ∈ {i-2, i-1, i}, với 0 <= start <= n-3
        return min(1 + dp(start + 3) for start in range(max(0, i - 2), min(i, n - 3) + 1))

    return dp(0)

print(min_moves_dp("XXX"), min_moves_dp("XXOX"), min_moves_dp("OOOO"), min_moves_dp("OXOOXXOOX"))

1 2 0 3


`@lru_cache` là **decorator** ghi nhớ kết quả của hàm theo đối số: gọi `dp(i)` lần hai trả ngay giá trị cũ (memoization) → DP chạy `O(n)` trạng thái.

## 10. Cách 4 — BFS vét cạn trên trạng thái (oracle cho chuỗi ngắn)

Hoàn toàn không dùng suy luận: coi mỗi chuỗi là một **trạng thái**, mỗi move (mọi vị trí bắt đầu `0..n−3`) là một **cạnh**. BFS từ `s` tìm số bước ít nhất tới chuỗi toàn `'O'`. Chỉ dùng được cho `n` nhỏ (số trạng thái tới `2^n`), nhưng là **oracle** đáng tin để test greedy.

`collections.deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu trong `O(1)`.

In [13]:
from collections import deque

def min_moves_bfs(s):
    n = len(s)
    target = 'O' * n
    dist = {s: 0}
    q = deque([s])
    while q:
        cur = q.popleft()
        if cur == target:
            return dist[cur]
        for start in range(n - 2):
            nxt = cur[:start] + 'OOO' + cur[start + 3:]
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                q.append(nxt)
    return -1

print(min_moves_bfs("XXX"), min_moves_bfs("XXOX"), min_moves_bfs("OOOO"), min_moves_bfs("OXOOXXOOX"))

1 2 0 3


## 11. Cách 5 — Một dòng với regex

`re.findall(r"X.{0,2}", s)` quét từ trái: gặp `X` thì "ăn" thêm tối đa 2 ký tự, rồi tiếp tục **sau** đoạn đã ăn — y hệt greedy `i += 3`. Số khớp = số move.

In [14]:
def min_moves_regex(s):
    return len(re.findall(r"X.{0,2}", s))

for t in ["XXX", "XXOX", "OOOO", "OXOOXXOOX", "XOXOXOX"]:
    print(f"{t!r:<14} khớp: {re.findall(r'X.{0,2}', t)} -> {min_moves_regex(t)}")

'XXX'          khớp: ['XXX'] -> 1
'XXOX'         khớp: ['XXO', 'X'] -> 2
'OOOO'         khớp: [] -> 0
'OXOOXXOOX'    khớp: ['XOO', 'XXO', 'X'] -> 3
'XOXOXOX'      khớp: ['XOX', 'XOX'] -> 2


## 12. Vì sao greedy đúng? (exchange argument + invariant)

**Invariant:** khi vòng lặp ở vị trí `i`, mọi `'X'` ở vị trí `< i` đã được phủ bởi `moves` move, và `moves` là **ít nhất** có thể để phủ các `'X'` ở `< i`, đồng thời phủ **xa nhất** có thể sang phải.

**Lập luận trao đổi:** xét một lời giải tối ưu bất kỳ `OPT`. Gọi `i` là X trái nhất. `OPT` phải có một move `M` phủ `i`, bắt đầu tại `p ≤ i`. Thay `M` bằng move bắt đầu tại `i` (hoặc `n−3` nếu sát cuối): các ô `< i` mà `M` phủ thêm đều là `'O'` (vì `i` là X trái nhất) nên không mất gì; còn phía phải, move mới phủ `{i, i+1, i+2}` ⊇ phần bên phải `i` của `M`. → Lời giải mới vẫn hợp lệ, **cùng số move**. Lặp lại cho phần còn lại → greedy đạt đúng số move của `OPT`.

**Kết thúc:** `i` tăng ít nhất 1 mỗi vòng → dừng sau tối đa `n` vòng.

## 13. Time / Space và bảng so sánh

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng trên list | tìm X đầu, sửa 3 ô | `O(n²)` | `O(n)` | Trực giác |
| 2. Greedy nhảy chỉ số | `i += 3` khi gặp X | `O(n)` | `O(1)` | **Khuyến nghị** |
| 3. DP + memo | `dp[i]` | `O(n)` | `O(n)` | Kiểm chứng tối ưu |
| 4. BFS trạng thái | vét cạn | `O(2^n · n²)` | `O(2^n · n)` | Chỉ là oracle cho n nhỏ |
| 5. Regex | `findall("X.{0,2}")` | `O(n)` | `O(n)` (list khớp) | Một dòng |

## 14. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. Biên: `n = 3` các dạng; X ở **cuối** (`"OOX"`, `"OOOOX"`).
3. Phản ví dụ "đếm số X rồi chia 3": `"XOOX"` có 2 X → `ceil(2/3) = 1` nhưng cần **2**.
4. Phản ví dụ "đếm số cụm X liên tiếp": `"XOX"` có 2 cụm nhưng chỉ cần **1**.
5. Đối chiếu ngẫu nhiên greedy / mô phỏng / DP / regex với BFS (n ≤ 10).

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual})")

f = min_moves
check("đề 1", f("XXX"), 1)
check("đề 2", f("XXOX"), 2)
check("đề 3", f("OOOO"), 0)
check("n=3, X ở cuối", f("OOX"), 1)
check("X sát cuối", f("OOOOX"), 1)
check("XOOX (không phải ceil(X/3))", f("XOOX"), 2)
check("XOX (không phải số cụm)", f("XOX"), 1)
check("7 X liên tiếp", f("X" * 7), 3)
check("1000 ký tự O", f("O" * 1000), 0)

đề 1                           PASS (1)
đề 2                           PASS (2)
đề 3                           PASS (0)
n=3, X ở cuối                  PASS (1)
X sát cuối                     PASS (1)
XOOX (không phải ceil(X/3))    PASS (2)
XOX (không phải số cụm)        PASS (1)
7 X liên tiếp                  PASS (3)
1000 ký tự O                   PASS (0)


In [16]:
import random
from itertools import product

# Vét toàn bộ chuỗi độ dài 3..9 (2^3 + ... + 2^9 chuỗi), so với BFS
total = 0
for n in range(3, 10):
    for tup in product("XO", repeat=n):
        t = "".join(tup)
        exp = min_moves_bfs(t)
        assert min_moves(t) == exp and min_moves_simulate(t) == exp
        assert min_moves_dp(t) == exp and min_moves_regex(t) == exp
        assert Solution().minimumMoves(t) == exp
        total += 1
print(f"Vét cạn {total} chuỗi độ dài 3..9: mọi cách khớp BFS")

rng = random.Random(2027)
for _ in range(500):
    t = "".join(rng.choice("XO") for _ in range(rng.randint(3, 200)))
    assert min_moves(t) == min_moves_dp(t) == min_moves_regex(t) == min_moves_simulate(t)
print("500 chuỗi ngẫu nhiên dài tới 200: greedy = DP = regex = mô phỏng")

Vét cạn 1016 chuỗi độ dài 3..9: mọi cách khớp BFS
500 chuỗi ngẫu nhiên dài tới 200: greedy = DP = regex = mô phỏng


`itertools.product("XO", repeat=n)` sinh **mọi** bộ `n` ký tự từ `{X, O}` — tổng `2^n` chuỗi. Đây là cách kiểm thử **toàn diện** khi miền nhỏ.

## 15. Lỗi người mới hay gặp

### Lỗi 1: `ceil(số X / 3)`

In [17]:
import math

def wrong_ceil(s):
    return math.ceil(s.count('X') / 3)

print("sai :", wrong_ceil("XOOX"))   # 1
print("đúng:", min_moves("XOOX"))    # 2

sai : 1
đúng: 2


**Vì sao sai:** move phủ 3 ô **liên tiếp**; hai X xa nhau không thể chung một move. **Sửa:** greedy theo vị trí.

### Lỗi 2: Đếm số cụm X liên tiếp

In [18]:
def wrong_groups(s):
    return len([g for g in s.split('O') if g])      # mỗi cụm X một move?

print("sai :", wrong_groups("XOX"), wrong_groups("XXXX"))   # 2 1
print("đúng:", min_moves("XOX"), min_moves("XXXX"))         # 1 2

sai : 2 1
đúng: 1 2


**Vì sao sai:** một move có thể phủ **hai cụm** (`XOX`), và một cụm dài có thể cần **nhiều** move (`XXXX`). (`s.split('O')` tách chuỗi tại mỗi `'O'`, cho ra các đoạn X.)

### Lỗi 3: Luôn `i += 1` (quên nhảy 3)

In [19]:
def wrong_step_one(s):
    moves = 0
    for c in s:
        if c == 'X':
            moves += 1
    return moves

print("sai :", wrong_step_one("XXX"))   # 3
print("đúng:", min_moves("XXX"))        # 1

sai : 3
đúng: 1


**Vì sao sai:** đếm mỗi X một move — bỏ qua việc một move phủ luôn 2 ô kế tiếp. **Sửa:** khi đặt move, nhảy `i += 3`.

### Lỗi 4: Dùng `for i in range(n)` rồi gán `i += 3` bên trong

In [20]:
def wrong_for_skip(s):
    moves = 0
    for i in range(len(s)):
        if s[i] == 'X':
            moves += 1
            i += 3          # vô tác dụng: vòng for gán lại i ở lượt sau
    return moves

print("sai :", wrong_for_skip("XXX"))   # 3
print("đúng:", min_moves("XXX"))

sai : 3
đúng: 1


**Vì sao sai:** trong Python, `for i in range(...)` lấy giá trị tiếp theo từ `range` mỗi vòng — sửa `i` bên trong **không** ảnh hưởng lượt sau. **Sửa:** dùng `while` và tự quản lý `i`.

## 16. Mẹo nhận dạng pattern và biến thể

**Nhận dạng:**
- "Mỗi thao tác phủ/xử lý một **đoạn liên tiếp độ dài cố định**, tìm số thao tác ít nhất để phủ hết các điểm" → **greedy**: điểm chưa phủ trái nhất → đặt đoạn **bắt đầu tại đó** (đẩy đoạn xa nhất về phải).
- Hint "delay as long as possible" = đặt quyết định muộn nhất mà vẫn hợp lệ.
- Duyệt có bước nhảy thay đổi → `while`, không `for`.

**Biến thể:**
1. Cửa sổ độ dài **`k`** thay vì 3: `i += k`.
2. **Phủ điểm trên trục số bằng đoạn độ dài L** (bài kinh điển "minimum number of intervals to cover points"): sort điểm, greedy giống hệt.
3. **LeetCode 452 — Minimum Number of Arrows to Burst Balloons:** greedy theo đầu mút phải.
4. **Flip thay vì gán** (LeetCode 995 — Minimum Number of K Consecutive Bit Flips): khó hơn vì flip có thể biến O thành X.

In [21]:
def min_moves_k(s, k):
    moves = i = 0
    while i < len(s):
        if s[i] == 'X':
            moves += 1
            i += k
        else:
            i += 1
    return moves

def cover_points(points, L):
    pts = sorted(points)
    count, covered_until = 0, float('-inf')
    for p in pts:
        if p > covered_until:            # điểm chưa phủ trái nhất
            count += 1
            covered_until = p + L        # đặt đoạn [p, p + L]
    return count

print("k=2, 'XXOX':", min_moves_k("XXOX", 2), "| k=4:", min_moves_k("XXOX", 4))
print("phủ điểm [1, 2, 6, 7, 11] bằng đoạn dài 2:", cover_points([1, 2, 6, 7, 11], 2))

k=2, 'XXOX': 2 | k=4: 1
phủ điểm [1, 2, 6, 7, 11] bằng đoạn dài 2: 3


## 17. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Trace greedy cho `s = "XOOXXOXOOOX"`: ghi `i`, quyết định, `moves`.

### Bài 2 — Dự đoán output
`re.findall(r"X.{0,2}", "OXXOXOOX")` trả về list nào? Đáp án bài toán là bao nhiêu?

### Bài 3 — Tự viết lời giải greedy

In [22]:
def exercise_min_moves(s: str) -> int:
    # TODO: while i < len(s): X -> moves += 1, i += 3; O -> i += 1
    pass

# assert exercise_min_moves("XXOX") == 2
# assert exercise_min_moves("OOOO") == 0

### Bài 4 — Biến thể: cửa sổ độ dài `k` (`1 <= k <= len(s)`)

In [23]:
def exercise_min_moves_k(s: str, k: int) -> int:
    # TODO: như bài chính nhưng nhảy k
    pass

# assert exercise_min_moves_k("XOXOX", 1) == 3
# assert exercise_min_moves_k("XOXOX", 5) == 1

### Bài 5 — Biến thể: trả về **danh sách vị trí bắt đầu** của các move (move phải nằm trong chuỗi: bắt đầu tối đa `n − 3`)

In [24]:
def exercise_move_positions(s: str) -> list[int]:
    # TODO: khi gặp X ở i, vị trí thực tế là min(i, n - 3)
    pass

# assert exercise_move_positions("OXOOXXOOX") == [1, 4, 6]
# assert exercise_move_positions("OOOO") == []

### Bài 6 — Mini challenge: phủ các điểm nguyên trên trục số bằng ít đoạn `[a, a + L]` nhất (điểm chưa sort, có thể trùng)

In [25]:
def exercise_cover_points(points: list[int], L: int) -> int:
    # TODO: sort; giữ covered_until; điểm nào > covered_until thì mở đoạn mới bắt đầu tại điểm đó
    pass

# assert exercise_cover_points([7, 1, 2, 11, 6], 2) == 3
# assert exercise_cover_points([5, 5, 5], 0) == 1

## 18. Gợi ý, đáp án tham khảo, template và checklist

### Gợi ý 3 mức
| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bắt đầu i = 0 là X | nhảy tới 3 (X) → 6 (X) → 9 (O) → 10 (X) | 4 move |
| 2 | Quét trái sang phải | khớp đầu tiên bắt đầu ở chỉ số 1 | `['XXO', 'XOO', 'X']` |
| 3 | Khuôn mục 6 | `while`, không `for` | `i += 3` / `i += 1` |
| 4 | Thay 3 bằng k | `i += k` khi gặp X | `k = 1` → đếm số X; `k = n` → 0 hoặc 1 |
| 5 | Ghi lại vị trí mỗi move | `n = len(s)` | `pos.append(min(i, n - 3))` |
| 6 | Sort trước | `if p > covered_until` | `covered_until = p + L` |

### Đáp án
- Bài 1: `"XOOXXOXOOOX"`: i=0 X → move 1, i=3; i=3 X → move 2, i=6; i=6 X → move 3, i=9; i=9 O → i=10; i=10 X → move 4, i=13 → **4**.
- Bài 2: `['XXO', 'XOO', 'X']` → **3** move.

In [26]:
def answer_min_moves(s):
    moves = i = 0
    while i < len(s):
        if s[i] == 'X':
            moves += 1
            i += 3
        else:
            i += 1
    return moves

def answer_min_moves_k(s, k):
    moves = i = 0
    while i < len(s):
        if s[i] == 'X':
            moves, i = moves + 1, i + k
        else:
            i += 1
    return moves

def answer_move_positions(s):
    n, pos, i = len(s), [], 0
    while i < n:
        if s[i] == 'X':
            pos.append(min(i, n - 3))
            i += 3
        else:
            i += 1
    return pos

def answer_cover_points(points, L):
    count, covered_until = 0, float('-inf')
    for p in sorted(points):
        if p > covered_until:
            count += 1
            covered_until = p + L
    return count

assert answer_min_moves("XOOXXOXOOOX") == 4
assert re.findall(r"X.{0,2}", "OXXOXOOX") == ['XXO', 'XOO', 'X'] and answer_min_moves("OXXOXOOX") == 3
assert answer_min_moves("XXOX") == 2 and answer_min_moves("OOOO") == 0
assert answer_min_moves_k("XOXOX", 1) == 3 and answer_min_moves_k("XOXOX", 5) == 1
assert answer_move_positions("OXOOXXOOX") == [1, 4, 6] and answer_move_positions("OOOO") == []
assert answer_cover_points([7, 1, 2, 11, 6], 2) == 3 and answer_cover_points([5, 5, 5], 0) == 1
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def minimumMoves(self, s: str) -> int:
        moves = i = 0
        while i < len(s):
            if s[i] == 'X':       # X trái nhất chưa phủ
                moves += 1        # đặt move bắt đầu tại đây
                i += 3            # nhảy qua 3 ô đã phủ
            else:
                i += 1
        return moves
```

Câu thần chú: **"Gặp X thì đặt move ngay tại đó, nhảy 3."**

### Checklist trước khi submit
- [ ] Dùng `while` (vì `i` nhảy 3), không sửa `i` trong `for`?
- [ ] Không dùng công thức `ceil(#X / 3)` hay đếm cụm?
- [ ] X sát cuối chuỗi vẫn được tính 1 move?
- [ ] Chuỗi không có X trả `0`?
- [ ] Đã test `"XOOX"` (2) và `"XOX"` (1)?